# GraMa experiments

Run > Run All Cells. The notebook installs what's missing, checks the data, builds the dataset,
trains every run of the chosen profile, and ends with the tables and figures for the paper.

| Profile | Data | What it's for | Time |
|---|---|---|---|
| `smoke` | synthetic CAN traffic | checking that everything works | ~5-10 min on a CPU |
| `quick` | CIC-IoV2024, benign capped at 150K rows | a first real look, tuning | ~30-60 min on a GPU |
| `full`  | CIC-IoV2024 | the paper numbers (3 seeds) | several hours on a GPU |

Finished runs are saved one by one to `results/<profile>/runs.jsonl`. If the kernel dies or the
server stops, run all cells again: finished runs are skipped. For `full`, a terminal is safer than
the notebook: `make full-background`, then `tail -f results/full.log`.

In [ ]:
PROFILE = "quick"   # "smoke", "quick" or "full"

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "config" / "experiments.yaml").exists(), "Open this notebook from inside the grama folder."

needed = {"torch": "torch", "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
          "scipy": "scipy", "yaml": "pyyaml", "tqdm": "tqdm", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import torch
from grama.utils.device import describe_device, pick_device

print("python", sys.version.split()[0], "| torch", torch.__version__)
print("device:", describe_device(pick_device()))
if not torch.cuda.is_available() and PROFILE != "smoke":
    print("No GPU found. This will work but be slow; ask for a GPU server if the hub offers one.")

## 2. Data

CIC-IoV2024 (Neto et al., *Internet of Things*, 2024) comes from
https://www.unb.ca/cic/datasets/iov-dataset-2024.html. The page asks for a name and email, then
links the download. Put the zip, or the folders inside it, in `data/raw/`; in JupyterLab you can
drag files into that folder in the file browser. Any folder layout works, zips are unpacked here,
and only the `decimal` CSVs are used.

In [ ]:
from grama.data.download import check
from grama.utils.config import Config

data_cfg = Config.from_yaml(ROOT / "config" / "data.yaml")
if PROFILE != "smoke":
    if not check(ROOT / data_cfg["dataset"]["raw_dir"], data_cfg["classes"]):
        print("\n" + "=" * 78)
        print("CIC-IoV2024 is not in data/raw yet, so this run switches to the SMOKE profile")
        print("(synthetic data). It shows the pipeline working; its numbers are not results.")
        print("Add the data and run all cells again to get the real ones.")
        print("=" * 78)
        PROFILE = "smoke"

In [ ]:
from grama.experiments.runner import Experiment

exp = Experiment(PROFILE, root=ROOT)
print(exp.profile.get("about", ""))
exp.prepare_data()
print("classes:", exp.meta["class_names"])
print("train sequences per class:", exp.meta["class_counts"]["train"])
print("test sequences per class: ", exp.meta["class_counts"]["test"])

## 3. Train

In [ ]:
plan = exp.plan()
done = exp.done_ids()
groups = {}
for s in plan:
    key = "ablation" if s.variant else ("poisoning" if s.attack else "clean")
    groups[key] = groups.get(key, 0) + 1
print(f"{len(plan)} runs in this profile ({groups}); {sum(s.run_id in done for s in plan)} already done.")

In [ ]:
exp.run_all()
efficiency = exp.efficiency()

## 4. Results

In [ ]:
from IPython.display import Markdown, display
from grama.experiments.report import make_report

summary = make_report(exp.out_dir)
rel = exp.out_dir.relative_to(ROOT).as_posix()
display(Markdown(summary.read_text().replace("](figures/", f"]({rel}/figures/")))

## Where everything is

- `results/<profile>/summary.md`: the page above
- `results/<profile>/tables/*.csv`: every table, for the paper
- `results/<profile>/figures/*.pdf`: figures for LaTeX (PNG copies to look at)
- `results/<profile>/runs.jsonl`: every run with its per-round history
- `results/<profile>/models/`: final global weights of the clean runs

To download everything, run `make pack PROFILE=quick` in a terminal and download
`results_quick.zip` from the file browser.